# Evaluation: the extraction flow on every sample set

Every upload goes through one flow (`src/one_flow.py`):

1. **Photo check.** A blurry, dark, or small photo gets a retake prompt (`upload_check.py`).
2. **Get the text out.** Textract for photos and scans (with the image), the file's own text for PDFs, Word files, and emails, Transcribe for voice, the rows for spreadsheets. Long inputs are read in pieces.
3. **Two models read each piece.** Nova Pro and Llama 4 Maverick each fill the same entry schema, citing the lines each entry came from (`model_readers.py`, `prompts.py`).
4. **Code cleans up what they read.** Dates such as "Tues" or "yesterday", CMID formats, matter names, units.
5. **Rule checks on every entry.** Hours on the cited lines, likely lost decimal points, unknown or ambiguous clients, a CMID that belongs to another client, missing or mismatched dates, hours on the page that no entry uses, and where the two models disagree.
6. **Two tiers.** A flagged entry goes to a person (`review.py`); the rest go to matching. The rule readers (`extract.py`) only read an upload when the model call fails, and those entries go to a person.

This notebook reads every sample set three ways and grades each against its answer key:
- **Rules**: the rule readers alone (photos and voice only, for reference).
- **Routes**: the previous design, where each input type had its own reader.
- **One flow**: the flow above.

**About the samples.** All sample data is made up: notes written by the team and two outside writers, synthetic typed files, and scripted voice memos. Labels say which sets are ours, synthetic, or from outside writers. The flow was developed on these samples, so these numbers are better than what new data would show.

**Running it.** Every model answer is saved in `data/model_answers/`, so the notebook runs with no AWS calls (`RUN_MODEL = False`). Section 8 needs matching's encoder (`RUN_MATCHING = True`). Photos and voice memos aren't in the repository, so those sets only run where they are on disk.

## 1. Setup

In [1]:
import json
import sys
from dataclasses import dataclass, field
from pathlib import Path

import pandas as pd

sys.path.insert(0, "../src")
sys.path.insert(0, "../../matching/src")
from benchmark import BILL_OUTCOMES, FIELDS, bill_outcome, mcnemar, model_cost, pair_found, with_ranges
from extract import extract
from fuzzy import client_name_table, matter_name_table
from grade import case_outcome, grade_photos, summarize
from held_out import build_key, load_cards, load_files, with_tk
from intake import for_matching, timekeeper_table
from messages import message_entries, read_eml, read_message
from model_first import read_photo
from model_readers import nova_rows, run_all
from one_flow import MAVERICK, ModelReader, firm_lookup, read
from preprocess import BAD_UPLOADS
from review import as_reviewed
from spreadsheets import read_spreadsheet
from tables import has_table
from textract import load_pages

pd.set_option("display.max_colwidth", 70)

In [2]:
DATA_PATH = Path("../../matching/data/synthetic_time_entries_v3.xlsx")
MATCHING_CONFIG = Path("../../matching/results/02_config.json")
RESULTS_DIR = Path("../../matching/results")
DATA_DIR = Path("../data")
TEXTRACT_DIR = DATA_DIR / "textract/dev"
HELD_TEXTRACT = DATA_DIR / "textract/held_out"
ANSWERS = DATA_DIR / "model_answers"
ONE_FLOW_DIR = ANSWERS / "one_flow"          # note photos read by Nova Pro, and scanned pages
ONE_FLOW_V4_DIR = ANSWERS / "one_flow_v4"    # Nova Pro on every input but note photos
SECOND_DIR = ANSWERS / "one_flow_second"     # Llama 4 Maverick, the second model
MAVERICK_PHOTOS = [ANSWERS / "open_models/llama_4_maverick_v2" / s for s in ["own", "w1"]]   # Maverick on note photos
PHOTO_ANSWERS = [ANSWERS / "nova_v2/nova_pro_photo", ANSWERS / "held_out/nova_pro_photo",
                 ANSWERS / "copies/nova_pro_photo"]   # Nova Pro on note photos
SHEETS_PATH = DATA_DIR / "dev_scans/2399_001.pdf"
COLUMN_NOTES = ["mon_fri_sample.jpg", "note_07.JPG"]
DOCUMENT_OWNER = {"log_02.docx": "TK106", "log_03.docx": "TK106", "shared_03.txt": "TK107"}
UPLOAD_DATE, UPLOADER, HELD_OUT_UPLOAD_DATE, YEAR = "2026-11-02", "TK102", "2026-11-13", 2026
TEAM = {"A0", "A0p", "A0n"}
REGION = "us-west-2"
PRICES = {"Nova Pro": (0.80, 3.20), "Llama 4 Maverick": (0.24, 0.97)}   # $ per million tokens in and out, Bedrock
RUN_MODEL = False      # True: ask Bedrock for pieces with no saved answer (section 2 says how many)
RUN_MATCHING = True    # section 8 needs matching's encoder
WORKERS = 4
RANGE_RESAMPLES = 1000

In [3]:
matter_master = pd.read_excel(DATA_PATH, sheet_name="Matter Master", dtype=str)
acronym_skip = json.loads(MATCHING_CONFIG.read_text())["acronym_skip"]
ref = {"client_names": client_name_table(matter_master, acronym_skip), "matter_names": matter_name_table(matter_master)}
timekeepers = timekeeper_table(pd.read_csv(DATA_DIR / "timekeepers.csv"))
aliases = matter_master["Known Aliases"].dropna().str.split(",").explode().str.strip()
clients = sorted(set(matter_master["Client"]) | set(aliases) - {"", "Firm"})
cmids = set(matter_master["CMID"])
common = dict(ref=ref, timekeepers=timekeepers)

**Answer keys.** One row per entry, written from what each sample says (`data/answer_keys/`), plus the outside writers' cards (`data/held_out/cards.csv`), and the typed notes and stress cases in `tests/cases/`.

In [4]:
def dated(key, col="work_date", fill=None):
    key[col] = pd.to_datetime(key[col].fillna(fill) if fill else key[col])
    return key


photos_key = dated(pd.read_csv(DATA_DIR / "answer_keys/notes.csv", dtype={"cmid": str}), fill=UPLOAD_DATE)
own_photos = list(photos_key["photo"].unique())
intended = pd.read_csv(DATA_DIR / "answer_keys/intended_matters.csv", dtype={"intended_cmid": str})
intended = intended.rename(columns={"source": "photo"})[["photo", "entry", "intended_cmid"]]

sheets_all = pd.read_csv(DATA_DIR / "answer_keys/sheets.csv", dtype={"cmid": str})
def sheet_key(sources):
    key = sheets_all[sheets_all["source"].isin(sources)].rename(columns={"source": "photo"}).copy()
    key["entry"] = key.groupby("photo").cumcount() + 1
    return dated(key)[["photo", "entry", "work_date", "hours", "client", "cmid", "narrative"]].assign(note="")
sheets_key, columns_key = sheet_key([SHEETS_PATH.name]), sheet_key(COLUMN_NOTES)

documents_key = dated(pd.read_csv(DATA_DIR / "answer_keys/documents.csv", dtype={"cmid": str}))
documents_key = documents_key.rename(columns={"file": "photo"}).drop(columns=["tk_id"])
spreadsheets_key = dated(pd.read_csv(DATA_DIR / "answer_keys/spreadsheets.csv", dtype={"cmid": str}))
spreadsheets_key = spreadsheets_key.rename(columns={"file": "photo"}).drop(columns=["tk_id"])

messages_key = dated(pd.read_csv(DATA_DIR / "answer_keys/messages.csv", dtype={"cmid": str}))
messages_key["photo"] = messages_key["message"] + "/" + messages_key["part"]
messages_key = messages_key.drop(columns=["message", "part", "tk_id"]).assign(narrative=None)

typed_cases = json.loads(Path("../tests/cases/typed_text.json").read_text())
typed = [(c, "TEXT") for c in typed_cases["typed_notes"]] + [(c, "VOICE") for c in typed_cases["dictated"]]
typed_key = pd.DataFrame([{"photo": f"{kind}: {c['layout']}", "entry": n + 1, "work_date": pd.Timestamp(e["work_date"]),
                           "hours": e["hours"], "client": e["client"], "cmid": e["cmid"], "narrative": None, "note": ""}
                          for c, kind in typed for n, e in enumerate(c["expected"])])
stress_cases = json.loads(Path("../tests/cases/stress_cases.json").read_text())

cards = load_cards(DATA_DIR / "held_out/cards.csv")
held_files = load_files(DATA_DIR / "held_out/files.csv")
held_key = build_key(cards, held_files, HELD_OUT_UPLOAD_DATE)
held_photos = {w: list(held_files[(held_files["form"] == "photo") & (held_files["writer"] == w)]["file"])
               for w in ["W1", "W2"]}
memos = held_files[held_files["form"] == "voice"].reset_index(drop=True)
last_date = cards.dropna(subset=["date"]).groupby("page")["date"].max()
recorded = {m.file: last_date.get(m.page, HELD_OUT_UPLOAD_DATE) for m in memos.itertuples()}
transcripts = {f: json.loads((DATA_DIR / "transcripts/held_out/plain" / f"{Path(f).stem}.json").read_text())["text"]
               for f in memos["file"]}
tk_of = held_key.groupby("photo")["tk_id"].first()

## 2. The readers

Nova Pro reads note photos with the photo and Textract's lines (`model_readers.system_prompt`), and every other input with its own instructions and worked examples (`prompts.py`). Llama 4 Maverick reads every piece too. Each answer is saved the first time it's asked for, one JSON file per piece.

In [5]:
def reuse(source, piece):
    """The saved Nova Pro answer for a note photo."""
    if piece is None and Path(str(source)).suffix.lower() in {".jpg", ".jpeg", ".png"}:
        for path in [folder / f"{Path(str(source)).stem}.json" for folder in PHOTO_ANSWERS] + [
                ONE_FLOW_DIR / f"{Path(str(source)).name}.json"]:
            if path.exists():
                return path
    return None


def reuse_second(source, piece):
    """The saved Llama 4 Maverick answer for a note photo."""
    if piece is None and Path(str(source)).suffix.lower() in {".jpg", ".jpeg", ".png"}:
        for folder in MAVERICK_PHOTOS:
            if (folder / f"{Path(str(source)).stem}.json").exists():
                return folder / f"{Path(str(source)).stem}.json"
    return None


@dataclass
class LoggedReader(ModelReader):
    """A ModelReader that keeps every answer it used, and the pieces it had no answer for."""
    used: list = field(default_factory=list)
    missing: list = field(default_factory=list)

    def answer(self, piece, source, number):
        path = self.path(source, number)
        if not path.exists() and self.runtime is None:
            self.missing.append((source, len(piece["lines"]), piece["photo"] is not None))
            return None
        found = super().answer(piece, source, number)
        self.used.append(found)
        return found


def new_reader(runtime=None):
    """The flow's reader: Nova Pro, with Llama 4 Maverick as the second model."""
    firm = {k: v for k, v in firm_lookup(matter_master).items() if k in ("client_of", "aliases_of")}
    model = LoggedReader(clients=clients, cmids=cmids, runtime=runtime, answers_dir=ONE_FLOW_V4_DIR,
                         answer_path=reuse, images_dir=ONE_FLOW_DIR / "pages", **firm)
    model.second = LoggedReader(clients=clients, cmids=cmids, runtime=runtime, answers_dir=SECOND_DIR,
                                answer_path=reuse_second, images_dir=ONE_FLOW_DIR / "pages", model_id=MAVERICK,
                                force_tool=False, open_note_prompt=True, **firm)
    return model


MAIN = "One flow"

## 3. Reading every input

Each sample set is read by **routes** (the previous design) and by **one flow**. Photos and voice also show the **rules** alone.

In [6]:
def one(model):
    return lambda content, input_type, image=None, **kw: read(content, input_type, ref, timekeepers, model,
                                                               image=image, **kw)


def photo_pages(folder, name):
    return load_pages(folder, name, "note") or load_pages(folder, name, "sheet")


def photo_jobs(names, textract_dir, photo_dir, tk, upload_date, model):
    """(name, {reader: entries}) jobs for note photos."""
    def job(n):
        note, sheet = photo_pages(textract_dir, n), load_pages(textract_dir, n, "sheet")
        saved_answer = reuse(n, None)
        answer = json.loads(saved_answer.read_text()) if saved_answer else None
        kw = dict(tk_id=tk(n), upload_date=upload_date, source=n)
        return n, {"Rules": extract(sheet if has_table(sheet) else note, "NOTE", **common, **kw),
                   "Routes": read_photo(n, note, sheet, answer, ref, timekeepers, cmids, tk(n), upload_date),
                   "One flow": read(note, "NOTE", ref, timekeepers, model, image=photo_dir / n, **kw)}
    return [lambda n=n: job(n) for n in names]


def gather(results):
    """{reader: entries with a photo column} from (name, {reader: entries}) results."""
    readers = results[0][1].keys() if results else []
    return {r: pd.concat([found[r].assign(photo=n) for n, found in results], ignore_index=True) for r in readers}


def read_everything(model):
    """{set: (key, {reader: entries}, label)} for every sample set, and the stress cases' outcomes."""
    sets, read_one = {}, one(model)
    sets["Note photos"] = (photos_key, gather(run_all(photo_jobs(
        own_photos, TEXTRACT_DIR, DATA_DIR / "dev_photos", lambda n: UPLOADER, UPLOAD_DATE, model), WORKERS)), "ours")
    sets["Notes with drawn columns"] = (columns_key, gather(run_all(photo_jobs(
        COLUMN_NOTES, TEXTRACT_DIR, DATA_DIR / "dev_photos", lambda n: UPLOADER, UPLOAD_DATE, model), WORKERS)), "ours")
    for w in ["W1", "W2"]:
        key = held_key[held_key["photo"].isin(held_photos[w])].drop(columns=["tk_id", "intended_cmid"])
        sets[f"Note photos, {w}"] = (key, gather(run_all(photo_jobs(
            held_photos[w], HELD_TEXTRACT, DATA_DIR / "held_out/photos", lambda n: tk_of[n], HELD_OUT_UPLOAD_DATE,
            model), WORKERS)), "held out (seen)")

    copies, copies_textract = DATA_DIR / "copies", DATA_DIR / "textract/copies"
    names = [(n, copies / f"{Path(n).stem}_{v}.jpg") for v in BAD_UPLOADS for n in own_photos]
    if all(load_pages(copies_textract, c, "note") for _, c in names):
        def copy_job(n, c):
            pages = load_pages(copies_textract, c, "note")
            saved_answer = reuse(c.name, None)
            answer = json.loads(saved_answer.read_text()) if saved_answer else None
            kw = dict(tk_id=UPLOADER, upload_date=UPLOAD_DATE)
            return c.stem, {"Rules": extract(pages, "NOTE", **common, source=n, **kw).assign(original=n),
                            "Routes": read_photo(n, pages, [], answer, ref, timekeepers, cmids, UPLOADER,
                                                       UPLOAD_DATE).assign(original=n),
                            "One flow": read(pages, "NOTE", ref, timekeepers, model, image=c, source=c.name,
                                             **kw).assign(original=n)}
        found = gather(run_all([lambda n=n, c=c: copy_job(n, c) for n, c in names], WORKERS))
        copy_key = pd.concat([photos_key[photos_key["photo"] == n].assign(photo=f"{Path(n).stem}_{v}")
                              for v in BAD_UPLOADS for n in own_photos], ignore_index=True)
        sets["Note photos, bad copies"] = (copy_key, found, "ours")
    else:
        print("No bad copies on this machine, so that set is left out")

    sheet_pages = load_pages(TEXTRACT_DIR, SHEETS_PATH, "sheet")
    kw = dict(source=SHEETS_PATH.name, year=YEAR)
    sets["Scanned sheets"] = (sheets_key, {
        "Routes": extract(sheet_pages, "SHEET", **common, **kw).assign(photo=SHEETS_PATH.name),
        "One flow": read(sheet_pages, "SHEET", ref, timekeepers, model, image=SHEETS_PATH, **kw).assign(
            photo=SHEETS_PATH.name)}, "ours")

    def document_job(n):
        kind = "PDF" if n.endswith(".pdf") else "DOC"
        kw = dict(tk_id=DOCUMENT_OWNER.get(n), upload_date="2026-11-04", source=n, year=YEAR)
        path = DATA_DIR / "dev_typed/documents" / n
        return n, {"Routes": extract(path, kind, **common, **kw),
                   "One flow": read(path, kind, ref, timekeepers, model, **kw)}
    sets["Typed documents"] = (documents_key, gather(run_all(
        [lambda n=n: document_job(n) for n in documents_key["photo"].unique()], WORKERS)), "synthetic")

    def spreadsheet_job(n):
        raw = read_spreadsheet(DATA_DIR / "dev_typed/spreadsheets" / n)
        kw = dict(source=n, upload_date="2026-11-02", year=YEAR)
        return n, {"Routes": extract(raw, "SPREADSHEET", **common, **kw),
                   "One flow": read(raw, "SPREADSHEET", ref, timekeepers, model, **kw)}
    sets["Spreadsheets"] = (spreadsheets_key, gather(run_all(
        [lambda n=n: spreadsheet_job(n) for n in spreadsheets_key["photo"].unique()], WORKERS)), "synthetic")

    messages = [read_message(p, DATA_DIR) for p in sorted((DATA_DIR / "dev_typed/messages").glob("msg_*.json"))]
    messages += [read_eml(p, DATA_DIR / "dev_typed/messages/attachments")
                 for p in sorted((DATA_DIR / "dev_typed/messages").glob("msg_*.eml"))]
    def message_job(m):
        before = message_entries(m, ref, timekeepers, TEXTRACT_DIR)[0]
        after = message_entries(m, ref, timekeepers, TEXTRACT_DIR, read=read_one)[0]
        return m["id"], {"Routes": before, "One flow": after}
    found = {}
    for reader, entries in gather(run_all([lambda m=m: message_job(m) for m in messages], WORKERS)).items():
        entries = entries.assign(photo=entries["message"] + "/" + entries["part"])
        found[reader] = entries[entries["photo"].isin(set(messages_key["photo"]))]
    sets["Emails with attachments"] = (messages_key, found, "synthetic")

    def typed_job(c, kind):
        name = f"{kind}: {c['layout']}"
        kw = dict(tk_id=UPLOADER, upload_date=UPLOAD_DATE, source=name)
        return name, {"Routes": extract(c["text"], kind, **common, **kw),
                      "One flow": read(c["text"], kind, ref, timekeepers, model, **kw)}
    sets["Typed and dictated notes"] = (typed_key, gather(run_all(
        [lambda c=c, k=k: typed_job(c, k) for c, k in typed], WORKERS)), "ours")

    def memo_job(f):
        year = pd.Timestamp(recorded[f]).year
        kw = dict(tk_id=tk_of[f], upload_date=recorded[f], source=f)
        nova = DATA_DIR / "model_answers/held_out/nova_pro_voice/plain" / f"{Path(f).stem}.json"
        before = (with_tk(nova_rows(json.loads(nova.read_text()), f, recorded[f], cmids, input_type="VOICE"),
                          tk_of[f], timekeepers, year) if nova.exists() else None)
        return f, {"Rules": extract(transcripts[f], "VOICE", **common, **kw),
                   "Routes": before if before is not None else extract(transcripts[f], "VOICE", **common, **kw),
                   "One flow": read(transcripts[f], "VOICE", ref, timekeepers, model, **kw)}
    for label, team in [("Voice memos, team (scripted)", True), ("Voice memos, outside", False)]:
        files = [f for f in memos["file"] if (memos.set_index("file").loc[f, "writer"] in TEAM) == team]
        if files:
            key = held_key[held_key["photo"].isin(files)].drop(columns=["tk_id", "intended_cmid"])
            sets[label] = (key, gather(run_all([lambda f=f: memo_job(f) for f in files], WORKERS)),
                           "ours" if team else "held out (seen)")

    def stress_job(n, case):
        kw = dict(tk_id="TK102", upload_date="2026-11-02", source=f"stress_{n:02d}")
        return {"Routes": case_outcome(as_reviewed(extract(case["text"], case["input"], **common, **kw)),
                                             case["expected"]),
                "One flow": case_outcome(as_reviewed(read(case["text"], case["input"], ref, timekeepers, model,
                                                          **kw)), case["expected"])}
    stress = pd.DataFrame(run_all([lambda n=n, c=c: stress_job(n, c) for n, c in enumerate(stress_cases, 1)],
                                  WORKERS))
    return sets, stress

In [7]:
runtime = None
if RUN_MODEL:
    import boto3
    from botocore.config import Config
    # a page read with its image can take over a minute, so a call may wait 5 minutes
    runtime = boto3.client("bedrock-runtime", region_name=REGION, config=Config(
        connect_timeout=20, read_timeout=300, retries={"max_attempts": 4, "mode": "adaptive"}))
model = new_reader(runtime)
model.catch_errors = model.second.catch_errors = True
sets, stress = read_everything(model)
print(f"Nova Pro: {len(model.used)} pieces read, {len(model.missing)} with no saved answer | "
      f"Llama 4 Maverick: {len(model.second.used)} read, {len(model.second.missing)} with no saved answer")
fell_back = {s: int(found[MAIN]["check_reasons"].str.contains("read_by_rules").sum()) for s, (_, found, _) in sets.items()}
print("Entries read by the rules because the model had no answer:", {s: n for s, n in fell_back.items() if n} or "none")

Nova Pro: 234 pieces read, 0 with no saved answer | Llama 4 Maverick: 234 read, 0 with no saved answer
Entries read by the rules because the model had no answer: none


## 4. Results

Percent of answer-key entries, with 95% bootstrap ranges over photos or files. "p" is McNemar's exact test of one flow against routes on the same entries. An entry is **right** when its client, CMID, date, and hours all match the key; **silent errors** are wrong or missing entries with no flag.

In [8]:
graded = {s: {r: grade_photos(as_reviewed(found[r]), key) for r in found} for s, (key, found, _) in sets.items()}
label_of = {s: label for s, (_, _, label) in sets.items()}

table = {}
for s, by_reader in graded.items():
    for r, g in by_reader.items():
        row = with_ranges(g, n=RANGE_RESAMPLES)
        row["sample"] = label_of[s]
        if r == MAIN and "Routes" in by_reader:
            row["p, right with no flag"] = mcnemar(by_reader["Routes"], g, "right, no flag")["p"]
            row["p, silent errors"] = mcnemar(by_reader["Routes"], g, "silent error")["p"]
        table[(s, r)] = row
results = pd.DataFrame(table).T
cols = ["sample", "entries", "found %", "right, no flag %", "right, no flag % range", "to review %", "silent errors %",
        "silent errors % range", "extras", "p, right with no flag", "p, silent errors"]
results = results[[c for c in cols if c in results]]
for c in [c for c in results if c.endswith("%")]:
    results[c] = results[c].astype(float).round().astype("Int64")
results.index.names = ["input", "reader"]
results

sample entries  found %  \
input                        reader                                       
Note photos                  Rules                ours      31      100   
                             Routes               ours      31      100   
                             One flow             ours      31      100   
Notes with drawn columns     Rules                ours      10      100   
                             Routes               ours      10      100   
                             One flow             ours      10      100   
Note photos, W1              Rules     held out (seen)      19       79   
                             Routes    held out (seen)      19      100   
                             One flow  held out (seen)      19      100   
Note photos, W2              Rules     held out (seen)      21       52   
                             Routes    held out (seen)      21       57   
                             One flow  held out (seen)      21       95   
Note photos, bad copies      Rules                ours     155       86   
                             Routes               ours     155      100   
                             One flow             ours     155      100   
Scanned sheets               Routes               ours      23      100   
                             One flow             ours      23      100   
Typed documents              Routes          synthetic      39      100   
                             One flow        synthetic      39      100   
Spreadsheets                 Routes          synthetic      17      100   
                             One flow        synthetic      17      100   
Emails with attachments      Routes          synthetic      41      100   
                             One flow        synthetic      41      100   
Typed and dictated notes     Routes               ours      22      100   
                             One flow             ours      22      100   
Voice memos, team (scripted) Rules                ours      46       80   
                             Routes               ours      46       98   
                             One flow             ours      46      100   
Voice memos, outside         Rules     held out (seen)       4       75   
                             Routes    held out (seen)       4      100   
                             One flow  held out (seen)       4      100   

                                       right, no flag %  \
input                        reader                       
Note photos                  Rules                   23   
                             Routes                  61   
                             One flow                71   
Notes with drawn columns     Rules                   60   
                             Routes                  60   
                             One flow                10   
Note photos, W1              Rules                    5   
                             Routes                  63   
                             One flow                63   
Note photos, W2              Rules                   10   
                             Routes                  10   
                             One flow                52   
Note photos, bad copies      Rules                   20   
                             Routes                  52   
                             One flow                11   
Scanned sheets               Routes                  61   
                             One flow                43   
Typed documents              Routes                 100   
                             One flow                97   
Spreadsheets                 Routes                  88   
                             One flow                94   
Emails with attachments      Routes                  85   
                             One flow                85   
Typed and dictated notes     Routes                  91   
                             One flow 

In [9]:
counts = pd.DataFrame({(s, r): summarize(g) for s, by in graded.items() for r, g in by.items()}).T
counts.index.names = ["input", "reader"]
counts[["right, no flag", "to review", "silent errors"]].unstack("reader")

right, no flag              to review         \
reader                             One flow Routes Rules  One flow Routes   
input                                                                       
Emails with attachments                35.0   35.0   NaN      10.0    6.0   
Note photos                            22.0   19.0   7.0      14.0   15.0   
Note photos, W1                        12.0   12.0   1.0       8.0    4.0   
Note photos, W2                        11.0    2.0   2.0      12.0    9.0   
Note photos, bad copies                17.0   81.0  31.0     167.0   93.0   
Notes with drawn columns                1.0    6.0   6.0       9.0    3.0   
Scanned sheets                         10.0   14.0   NaN      13.0    9.0   
Spreadsheets                           16.0   15.0   NaN       1.0    1.0   
Typed and dictated notes               20.0   20.0   NaN       2.0    2.0   
Typed documents                        38.0   39.0   NaN       1.0    0.0   
Voice memos, outside                    2.0    3.0   0.0       2.0    1.0   
Voice memos, team (scripted)           30.0   25.0  14.0      15.0   15.0   

                                   silent errors               
reader                       Rules      One flow Routes Rules  
input                                                          
Emails with attachments        NaN           0.0    0.0   NaN  
Note photos                   23.0           0.0    2.0   1.0  
Note photos, W1               12.0           1.0    5.0   6.0  
Note photos, W2                9.0           1.0   11.0  12.0  
Note photos, bad copies       90.0           1.0   10.0  36.0  
Notes with drawn columns       3.0           0.0    1.0   1.0  
Scanned sheets                 NaN           1.0    0.0   NaN  
Spreadsheets                   NaN           0.0    1.0   NaN  
Typed and dictated notes       NaN           0.0    0.0   NaN  
Typed documents                NaN           0.0    0.0   NaN  
Voice memos, outside           0.0           0.0    1.0   4.0  
Voice memos, team (scripted)  12.0           1.0    6.0  20.0

**All sets together.** Each set counted once (bad copies left out, since they repeat our photos).

In [10]:
pooled_sets = [s for s in graded if s != "Note photos, bad copies"]
order = ["Routes", MAIN]
pooled = {r: pd.concat([graded[s][r].assign(photo=s + "/" + graded[s][r]["photo"].astype(str)) for s in pooled_sets],
                       ignore_index=True) for r in order}
total = pd.DataFrame({r: summarize(g) for r, g in pooled.items()}).T
cols = ["right, no flag", "right, flagged", "wrong, flagged", "wrong, silent", "missing", "extra, flagged",
        "extra, silent", "to review", "silent errors"]
display(total[cols])
print("p, right with no flag:", mcnemar(pooled["Routes"], pooled[MAIN], "right, no flag")["p"],
      "| p, silent errors:", mcnemar(pooled["Routes"], pooled[MAIN], "silent error")["p"])

,"right, no flag","right, flagged","wrong, flagged","wrong, silent",missing,"extra, flagged","extra, silent",to review,silent errors
Routes,190,29,27,17,10,9,0,65,27
One flow,197,34,38,3,1,15,0,87,4


p, right with no flag: 0.4 | p, silent errors: 1.5e-06


### 4a. Which review reasons catch errors

For each reason, how many entries it flags and how many of those were really wrong (or extra). "Alone" counts entries where it is the only reason. Bad copies are left out.

In [11]:
def reason_table(reader, bad_copies=False):
    rows = []
    for s, (key, found, _) in sets.items():
        if (s == "Note photos, bad copies") != bad_copies or reader not in found:
            continue
        g = graded[s][reader]
        for x in g[g["flagged"].fillna(False).astype(bool)].itertuples():
            reasons = [t for t in str(x.check).split("; ") if t and t != "nan"]
            rows += [{"reason": t, "caught": x.outcome != "right", "alone": len(reasons) == 1} for t in reasons]
    df = pd.DataFrame(rows, columns=["reason", "caught", "alone"])
    out = df.groupby("reason").agg(flags=("caught", "size"), caught=("caught", "sum"))
    alone = df[df["alone"]].groupby("reason").agg(alone=("caught", "size"), caught_alone=("caught", "sum"))
    return out.join(alone).fillna(0).astype(int).sort_values("flags", ascending=False)


reason_table(MAIN)

,flags,caught,alone,caught_alone
reason,,,,
readers_disagree,50,40,9,5
client_unknown,19,13,2,1
unused_hours,17,8,9,1
not_on_page,16,7,2,0
date_gap,12,11,2,1
hours_10x,12,12,1,1
model_unsure,12,10,0,0
client_gap,10,8,2,1
cmid_unknown,5,5,0,0


### 4b. The photo check at upload

`upload_check.check_photo` runs before Textract. In the app a photo that fails gets a retake prompt; one sent anyway gets `poor_photo` and goes to review. Below: the share of each kind of bad copy the check catches, and how many original photos it would wrongly ask to retake.

In [12]:
from upload_check import check_photo

copies = DATA_DIR / "copies"
if copies.exists():
    checked = pd.DataFrame([{"kind": v, "photo": n, **check_photo(copies / f"{Path(n).stem}_{v}.jpg")}
                            for v in BAD_UPLOADS for n in own_photos])
    display(checked.groupby("kind")[["ok"]].agg(lambda x: f"{(~x).sum()} of {len(x)} asked to retake"))
    originals = [(DATA_DIR / "dev_photos" / n) for n in own_photos] + [
        DATA_DIR / "held_out/photos" / n for w in ["W1", "W2"] for n in held_photos[w]]
    failed = [p.name for p in originals if p.exists() and not check_photo(p)["ok"]]
    print(f"Original photos asked to retake: {len(failed)} of {sum(p.exists() for p in originals)}", failed or "")
else:
    print("The photos aren't on this machine (they aren't in the repository), so this check is skipped")

,ok
kind,
blurry,17 of 17 asked to retake
dark,17 of 17 asked to retake
medium,17 of 17 asked to retake
small,17 of 17 asked to retake
tilted,0 of 17 asked to retake


Original photos asked to retake: 0 of 27 


## 5. Stress cases

58 hard written cases (`tests/cases/stress_cases.json`): run-together entries, odd units, corrections, notes that aren't time. The rule readers were written with these cases in view.

In [13]:
stress.apply(pd.Series.value_counts).fillna(0).astype(int)

,Routes,One flow
right,54,52
"wrong, flagged",4,6


## 6. Where one flow is wrong

Every entry one flow gets wrong with no flag, next to how routes read it, then how entries moved between the two.

In [14]:
cols = ["photo", "entry", "outcome", "wrong", "found", "check"]
silent = []
for s, by in graded.items():
    g = by[MAIN]
    wrong = g[g["outcome"].isin(["wrong, silent", "missing", "extra"]) & ~g["flagged"].fillna(False).astype(bool)]
    before = by["Routes"].dropna(subset=["entry"]).drop_duplicates(["photo", "entry"]).set_index(
        ["photo", "entry"])["outcome"] if "Routes" in by else None
    for row in wrong.itertuples():
        silent.append({"input": s, **{c: getattr(row, c, None) for c in cols},
                       "routes": None if before is None or pd.isna(row.entry) else
                       before.get((row.photo, row.entry))})
silent = pd.DataFrame(silent)
silent if len(silent) else "No silent errors"

,input,photo,entry,outcome,wrong,found,check,routes
0,"Note photos, W1",w1_A.JPG,3,"wrong, silent",client,Lantern None 1.4,,"wrong, silent"
1,"Note photos, W2",w2_E.JPG,3,missing,NaN,NaN,NaN,missing
2,"Note photos, bad copies",note_18_tilted,1,"wrong, silent",cmid,Harborlight None 1.0,,"wrong, silent"
3,Scanned sheets,2399_001.pdf,4,"wrong, silent",client,Halvorson Industrial 3012.004 3.0,,right
4,"Voice memos, team (scripted)",a0p_C.m4a,1,"wrong, silent",hours,nan 7215.003 0.7,,"wrong, silent"


In [15]:
fixed = []
for s, by in graded.items():
    if "Routes" not in by:
        continue
    a = by["Routes"].dropna(subset=["entry"]).set_index(["photo", "entry"])["outcome"]
    b = by[MAIN].dropna(subset=["entry"]).set_index(["photo", "entry"])["outcome"]
    both = pd.concat({"routes": a, "one flow": b}, axis=1).dropna()
    fixed.append(pd.crosstab(both["routes"], both["one flow"]).stack().rename(s))
pd.concat(fixed, axis=1).fillna(0).astype(int)

Note photos  Notes with drawn columns  \
routes         one flow                                                
right          right                    25                         1   
               wrong, flagged            0                         7   
wrong, flagged right                     0                         0   
               wrong, flagged            4                         1   
wrong, silent  right                     0                         0   
               wrong, flagged            2                         1   
right          wrong, silent             0                         0   
wrong, flagged wrong, silent             0                         0   
wrong, silent  wrong, silent             0                         0   
missing        missing                   0                         0   
               right                     0                         0   
               wrong, flagged            0                         0   
right          missing                   0                         0   
wrong, flagged missing                   0                         0   
wrong, silent  missing                   0                         0   
missing        wrong, silent             0                         0   

                               Note photos, W1  Note photos, W2  \
routes         one flow                                           
right          right                        13                7   
               wrong, flagged                0                0   
wrong, flagged right                         0                1   
               wrong, flagged                1                2   
wrong, silent  right                         0                2   
               wrong, flagged                4                0   
right          wrong, silent                 0                0   
wrong, flagged wrong, silent                 0                0   
wrong, silent  wrong, silent                 1                0   
missing        missing                       0                1   
               right                         0                8   
               wrong, flagged                0                0   
right          missing                       0                0   
wrong, flagged missing                       0                0   
wrong, silent  missing                       0                0   
missing        wrong, silent                 0                0   

                               Note photos, bad copies  Scanned sheets  \
routes         one flow                                                  
right          right                               110              15   
               wrong, flagged                        0               2   
wrong, flagged right                                 0               5   
               wrong, flagged                       39               0   
wrong, silent  right                                 0               0   
               wrong, flagged                        5               0   
right          wrong, silent                         0               1   
wrong, flagged wrong, silent                         0               0   
wrong, silent  wrong, silent                         1               0   
missing        missing                               0               0   
               right                                 0               0   
               wrong, flagged                        0               0   
right          missing                               0               0   
wrong, flagged missing                               0               0   
wrong, silent  missing                               0               0   
missing        wrong, silent                         0               0   

                               Typed documents  Spreadsheets  \
routes         one flow                                        
right          right                        39            16   
  

## 7. Cost and time

Per piece read (a photo, a page, a message part, up to 25 spreadsheet rows), from the saved answers.

In [16]:
costs = []
for name, m, price in [("Nova Pro", model, PRICES["Nova Pro"]), ("Llama 4 Maverick", model.second, PRICES["Llama 4 Maverick"])]:
    answers = [a for a in m.used if a is not None and "error" not in a]
    cost = model_cost(answers, price) if answers else {}
    tokens = pd.DataFrame([a["usage"] for a in answers]) if answers else pd.DataFrame()
    costs.append({"model": name, "pieces": len(answers), **cost,
                  "input tokens per piece": round(tokens["inputTokens"].mean()) if len(tokens) else None,
                  "total $": round(len(answers) * (cost.get("$ per page") or 0), 2)})
pd.DataFrame(costs).set_index("model")

,pieces,$ per page,seconds per page,input tokens per piece,total $
model,,,,,
Nova Pro,234,0.0039,2.2,4111,0.91
Llama 4 Maverick,234,0.0015,1.0,5462,0.35


## 8. Through matching

Every entry with an intended matter (our photos, drawn columns, the scanned sheet, the outside writers, and the voice memos) through matching's `decide()`, by routes and by one flow, ending in one of four places (`benchmark.bill_outcome`): billed right, to a person, billed wrong unseen, or lost.

In [17]:
def with_intended(key, s):
    if s.startswith(("Note photos, W", "Voice")):
        return key.merge(held_key[["photo", "entry", "intended_cmid"]], on=["photo", "entry"], how="left")
    return key.merge(intended, on=["photo", "entry"], how="left")


def through_matching(found, key_rows):
    from pipeline import decide
    found = found.reset_index(drop=True)
    _, _, decided = decide(for_matching(found), matching_ref)
    out = pair_found(as_reviewed(found), key_rows).merge(decided[["entry_id", "action", "pred_cmid"]], on="entry_id",
                                                         how="left")
    out = out.merge(key_rows[["photo", "entry", "intended_cmid"]].rename(columns={"intended_cmid": "intended"}),
                    on=["photo", "entry"], how="left")
    out["top_pick"] = out["pred_cmid"].astype(str).str.split(" | ", regex=False).str[0]
    out["bill"] = out.apply(bill_outcome, axis=1)
    return out


if RUN_MATCHING:
    from pipeline import load_reference
    from sentence_transformers import SentenceTransformer
    encoder = SentenceTransformer(json.loads(MATCHING_CONFIG.read_text())["embedding_model"])
    matching_ref = load_reference(DATA_PATH, RESULTS_DIR, encoder, "default")
    bills, unseen = {}, []
    for s, (key, found, _) in sets.items():
        if s.startswith(("Typed", "Spreadsheets", "Emails")) or s.endswith("bad copies"):
            continue
        key_rows = with_intended(key, s)
        key_rows = key_rows[key_rows["intended_cmid"].notna()]
        for r in ["Routes", MAIN]:
            out = through_matching(found[r][found[r]["photo"].isin(set(key_rows["photo"]))], key_rows)
            bills[(s, r)] = out["bill"].value_counts()
            unseen.append(out[out["bill"] == "billed wrong, unseen"].assign(input=s, reader=r))
    bills = pd.DataFrame(bills).T.reindex(columns=BILL_OUTCOMES).fillna(0).astype(int)
    bills.index.names = ["input", "reader"]
    display(bills)
    display(bills.groupby(level="reader").sum())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

bill                                   billed right  to a person  \
input                        reader                                
Note photos                  Routes               9           25   
                             One flow            10           25   
Notes with drawn columns     Routes               3            7   
                             One flow             0           10   
Note photos, W1              Routes              11           10   
                             One flow             9           12   
Note photos, W2              Routes               1           11   
                             One flow             9           13   
Scanned sheets               Routes               0           22   
                             One flow             0           24   
Voice memos, team (scripted) Routes              16           28   
                             One flow            20           25   
Voice memos, outside         Routes               2            3   
                             One flow             0            4   

bill                                   billed wrong, unseen  lost  
input                        reader                                
Note photos                  Routes                       2     0  
                             One flow                     1     0  
Notes with drawn columns     Routes                       0     0  
                             One flow                     0     0  
Note photos, W1              Routes                       0     0  
                             One flow                     0     0  
Note photos, W2              Routes                       1     9  
                             One flow                     1     1  
Scanned sheets               Routes                       1     0  
                             One flow                     0     0  
Voice memos, team (scripted) Routes                       1     1  
                             One flow                     1     0  
Voice memos, outside         Routes                       0     0  
                             One flow                     0     0

bill,billed right,to a person,"billed wrong, unseen",lost
reader,,,,
One flow,48,113,3,1
Routes,42,106,5,10


In [18]:
if RUN_MATCHING:
    unseen = pd.concat(unseen, ignore_index=True)
    display(unseen[["input", "reader", "photo", "entry", "intended", "top_pick", "hours_ok", "date_ok"]]
            if len(unseen) else "Nothing billed wrong unseen")

,input,reader,photo,entry,intended,top_pick,hours_ok,date_ok
0,Note photos,Routes,note_05.JPG,2.0,2960.003,2960.003,True,False
1,Note photos,Routes,note_12.jpg,1.0,3120.002,3120.001,True,True
2,Note photos,One flow,note_12.jpg,1.0,3120.002,3120.001,True,True
3,"Note photos, W2",Routes,w2_A.JPG,1.0,7222.002,7222.002,True,False
4,"Note photos, W2",One flow,w2_H.JPG,3.0,8012.002,8012.001,True,True
5,Scanned sheets,Routes,2399_001.pdf,NaN,NaN,4590.003,False,False
6,"Voice memos, team (scripted)",Routes,a0p_B.m4a,3.0,2960.003,2960.003,True,False
7,"Voice memos, team (scripted)",One flow,a0p_C.m4a,1.0,7215.003,7215.003,False,True


## Findings

Read from saved answers: 234 pieces each for Nova Pro and Llama 4 Maverick, none read by the rules, $1.26 for both models, about $0.005 a piece. The flow was developed on these samples, so new data will do worse; the outside writers' sets are labeled "held out (seen)" because they were read while the flow was built.

**All sets together (bad copies left out):**

| | Right, no flag | To review | Silent errors |
|---|---|---|---|
| Routes (a reader per input type) | 190 | 65 | 27 |
| One flow | 197 | 87 | 4 |

- **Silent errors fall from 27 to 4** (McNemar p < 0.001), and lost entries from 10 to 1. Right with no flag is about the same (p = 0.4). The cost is review: 87 entries against 65, because one flow flags its own mistakes instead of making them silently.
- **Handwriting gains most.** The outside writer with drawn columns (W2): silent errors 11 to 1, right with no flag 2 to 11. Our own photos: silent errors 2 to 0. The team's voice memos: silent errors 6 to 1.
- **Typed inputs stay close to the rule readers**, which were written on these samples: typed documents 38 right with no flag against 39, emails 35 against 35, spreadsheets 16 against 15, typed notes 20 against 20, none with a silent error.
- **The 4 silent errors left:** a short client name ("Lantern") that the check for ambiguous names misses; one entry lost on a W2 page; a voice memo where Transcribe heard 0.7 for 1.7, so both models read the wrong number; and a scanned-sheet entry where the answer key puts the matter name in the client column, so the reader is right and the key is off.
- **Through matching:** billed right 48 (routes 42), to a person 113 (106), billed wrong unseen 3 (5), lost 1 (10). Two of the 3 billed wrong are matching picking the wrong matter of the same client, on entries extraction read right; the third is the Transcribe error above.
- **The photo check** asks for a retake on every blurry, dark, medium, and small copy (17 of 17 each) and on none of the 27 original photos; tilted photos get through. On bad copies sent anyway, one flow has 1 silent error against routes' 10.
- **Stress cases:** 52 right and 6 flagged, with no silent errors (routes 54 right, 4 flagged).

**What these samples can't show:** long documents (the largest answer has 10 entries), a real firm's client list (this one has 35 clients), real audio, and natural handwriting with cross-outs. Narrative and timekeeper aren't graded. One flow is ready for a supervised pilot, where a person reviews what it flags.